In [15]:
import sys
from pathlib import Path
# import us_stock_valuation_preprocessing as stock_prep

def find_data_folder():
    """DATA 폴더 찾기"""
    current = Path.cwd()

    # 상위 5단계까지 검색
    for i in range(5):
        data_path = current / "DATA"
        if data_path.exists():
            target_files = [
                data_path / "stock_invest_function.py",
                data_path / "us_stock_valuation_preprocessing.py"
            ]
            if all(f.exists() for f in target_files):
                return data_path
        current = current.parent
    return None

def fix_and_import_modules():
    """모듈을 수정해서 import"""
    data_path = find_data_folder()
    if not data_path:
        print("DATA 폴더를 찾을 수 없습니다")
        return None, None

    sys.path.insert(0, str(data_path))
    print(f"DATA 폴더 발견: {data_path}")

    # stock_invest_function은 정상 import
    try:
        import stock_invest_function as stock_func
        print("stock_invest_function import 성공!")
    except Exception as e:
        print(f"stock_invest_function import 실패: {e}")
        stock_func = None

    # us_stock_valuation_preprocessing은 파일을 읽어서 수정 후 실행
    try:
        preprocessing_file = data_path / "us_stock_valuation_preprocessing.py"

        # 파일 내용 읽기
        with open(preprocessing_file, 'r', encoding='utf-8') as f:
            content = f.read()

        # 문제가 되는 부분을 찾아서 수정
        # "from stock_invest_function import *"를 모듈 최상위로 이동
        lines = content.split('\n')

        # import * 구문을 찾아서 제거하고 최상위에 추가
        fixed_lines = []
        import_lines = []
        inside_function = False

        for line in lines:
            # 함수 시작 감지
            if line.strip().startswith('def '):
                inside_function = True

            # import * 구문 처리
            if 'from stock_invest_function import *' in line:
                if inside_function:
                    # 함수 내부에 있으면 제거
                    continue
                else:
                    # 최상위에 있으면 그대로 유지
                    import_lines.append(line)
                    continue

            fixed_lines.append(line)

        # 수정된 내용 조합
        final_content = '\n'.join(import_lines + [''] + fixed_lines)

        # 수정된 내용을 실행해서 모듈 생성
        import types
        stock_prep = types.ModuleType('us_stock_valuation_preprocessing')

        # get_db_host 함수가 필요한 경우를 위해 미리 정의
        if stock_func and hasattr(stock_func, 'get_db_host'):
            stock_prep.get_db_host = stock_func.get_db_host
        else:
            stock_prep.get_db_host = lambda: 'localhost'

        # 수정된 코드 실행
        exec(final_content, stock_prep.__dict__)

        print("us_stock_valuation_preprocessing 수정 후 import 성공!")
        return stock_func, stock_prep

    except Exception as e:
        print(f"파일 수정 중 오류: {e}")
        return stock_func, None

# 실행
stock_func, stock_prep = fix_and_import_modules()

if stock_func and stock_prep:
    print("모든 모듈이 성공적으로 로드되었습니다!")
    print("사용 가능한 함수들:")
    if hasattr(stock_prep, 'run_data_preprocessing'):
        print("- stock_prep.run_data_preprocessing()")
    if hasattr(stock_prep, 'run_simple_preprocessing'):
        print("- stock_prep.run_simple_preprocessing()")
else:
    print("모듈 로드에 실패했습니다.")

import us_stock_valuation_preprocessing as stock_prep

DATA 폴더 발견: C:\Users\MetaM\PycharmProjects\stock_forecast\DATA
stock_invest_function import 성공!
us_stock_valuation_preprocessing 수정 후 import 성공!
모든 모듈이 성공적으로 로드되었습니다!
사용 가능한 함수들:
- stock_prep.run_data_preprocessing()
- stock_prep.run_simple_preprocessing()


In [16]:
from DATA.stock_invest_function import *

In [17]:

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# 1. 먼저 디버깅으로 원인 파악
# stock_prep.debug_missing_data(result, 'AMAT', db_info)

# 2. 개선된 버전으로 재실행
result_fixed = stock_prep.run_fixed_enhanced_preprocessing(
    ticker='GE',
    api_key=api_key,
    db_info=db_info,
    hs_code='841191'
)

result = result_fixed[result_fixed['date_month_end'] >= '2013-01-31']
# 단일 종목 간단 버전
# result = stock_prep.run_simple_preprocessing('AAPL', api_key)

NaN 문제 해결 버전 전처리 시작
대상 종목: GE
API 상태: API 연결 성공

1. FMP 데이터 수집
매출 데이터 수집 중...


Revenue:   0%|          | 0/1 [00:00<?, ?it/s]

   GE: 160개 분기


Revenue: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]


매출 데이터 수집 완료: 160 레코드
시가총액 데이터 수집 중...


Market Cap: 100%|██████████| 1/1 [00:18<00:00, 18.63s/it]

   GE: 189개 월
시가총액 데이터 수집 완료: 189 레코드

2. 개선된 DB 데이터 보완
매출 데이터 보완 중... (개선 버전)
DB 매출 데이터 조회 실패: (pymysql.err.ProgrammingError) (1146, "Table 'investar.fundq_df' doesn't exist")
[SQL: 
        SELECT date, ticker, saleq
        FROM fundq_df 
        WHERE ticker = 'GE' 
        AND saleq IS NOT NULL
        AND date <= '2024-12-31'
        ORDER BY date ASC
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)
DB 매출 데이터가 없어 보완 불가
시가총액 데이터 보완 중... (개선 버전)


DB 시가총액 데이터 조회 실패: (pymysql.err.ProgrammingError) (1146, "Table 'investar.fundm_df' doesn't exist")
[SQL: 
        SELECT date, ticker, me
        FROM fundm_df 
        WHERE ticker = 'GE' 
        AND me IS NOT NULL
        AND date <= '2024-12-31'
        ORDER BY date ASC
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)
DB 시가총액 데이터가 없어 보완 불가

3. 데이터 결합

4. 보수적인 결측치 보완
보수적인 결측치 보완 적용 중...
보수적인 결측치 보완 완료
데이터 결합 완료: 287건

5. TTM 및 PSR 계산
TTM 및 PSR 계산 중...
TTM 및 PSR 계산 완료: PSR 유효 데이터 189건

6. 수출 데이터 결합
수출 데이터 수집 중... (HS Code: 841191)
가장 최근 input_date: 2025-09-02
해당 날짜의 데이터: 165개
수출 데이터 수집 완료: 165 레코드

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
NaN 문제 해결 버전 전처리 완료!
최종 데이터: 299건

7. 2023년 8-9월 데이터 검증:
  2023-08-31: market_cap=98.87, revenue=8.75
  2023-09-30: market_cap=95.49, revenue=9.30


In [18]:
result

,ticker,date_month_end,market_cap_billions,revenue_billions,revenue_ttm,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr
134,GE,2013-01-31,138.26,36.76,144.46,144.46,142.41,0.970859,841191,104240000.0
135,GE,2013-02-28,144.09,36.76,145.75,145.75,143.17,1.006426,841191,99986000.0
136,GE,2013-03-31,143.47,33.33,143.61,143.61,144.46,0.993147,841191,97927000.0
137,GE,2013-04-30,136.84,33.33,140.18,140.18,145.75,0.938868,841191,112503000.0
138,GE,2013-05-31,143.17,33.33,136.75,136.75,143.61,0.996936,841191,82669500.0
...,...,...,...,...,...,...,...,...,...,...
294,NaN,2026-05-31,NaN,NaN,NaN,NaN,NaN,NaN,841191,294712000.0
295,NaN,2026-06-30,NaN,NaN,NaN,NaN,NaN,NaN,841191,296792000.0
296,NaN,2026-07-31,NaN,NaN,NaN,NaN,NaN,NaN,841191,264542000.0
297,NaN,2026-08-31,NaN,NaN,NaN,NaN,NaN,NaN,841191,301608000.0


In [19]:
from us_sarima_forecast import *

In [20]:
# 1. 각 모델별로 구분된 변수명 사용
# =================================================================

# SARIMA 외생변수 포함 예측
sarima_forecast_result = sarima_forecast_with_export(
    final_data=result,
    export_forecast_start_date="2025-10",
    USE_EXOGENOUS=True,
    forecast_months=12
)

# SARIMA 매출 예측 파이프라인
sarima_result_data, sarima_quarterly_data, sarima_forecast_result, sarima_model_info = revenue_sarima_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4
)

# LSTM 매출 예측 파이프라인
lstm_result_data, lstm_quarterly_data, lstm_forecast_result, lstm_model_info = revenue_lstm_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4,
    lookback_window=8,
    epochs=100
)

# Prophet 매출 예측 파이프라인
prophet_result_data, prophet_quarterly_data, prophet_forecast_results, prophet_model_infos = revenue_prophet_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4,
    use_exogenous=True,
    exog_cols=['expDlr']
)

# Exponential Smoothing 매출 예측 파이프라인
es_result_data, es_quarterly_data, es_forecast_result, es_model_info = revenue_es_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4
)

# 2. 모든 결과를 딕셔너리로 통합 관리
# =================================================================

# 모델별 결과 통합
all_forecast_results = {
    'sarima': {
        'result_data': sarima_result_data,
        'quarterly_data': sarima_quarterly_data,
        'forecast_result': sarima_forecast_result,
        'model_info': sarima_model_info
    },
    'lstm': {
        'result_data': lstm_result_data,
        'quarterly_data': lstm_quarterly_data,
        'forecast_result': lstm_forecast_result,
        'model_info': lstm_model_info
    },
    'prophet': {
        'result_data': prophet_result_data,
        'quarterly_data': prophet_quarterly_data,
        'forecast_result': prophet_forecast_results,
        'model_info': prophet_model_infos
    },
    'exponential_smoothing': {
        'result_data': es_result_data,
        'quarterly_data': es_quarterly_data,
        'forecast_result': es_forecast_result,
        'model_info': es_model_info
    }
}

# 3. 모든 예측 결과를 하나의 DataFrame으로 통합
# =================================================================

import pandas as pd
from datetime import datetime

def combine_all_forecasts(forecast_results_dict):
    """
    모든 모델의 예측 결과를 하나의 DataFrame으로 통합
    """
    combined_forecasts = []

    for model_name, results in forecast_results_dict.items():
        try:
            forecast_data = results['forecast_result']

            # forecast_result가 DataFrame인 경우
            if isinstance(forecast_data, pd.DataFrame):
                temp_df = forecast_data.copy()
                temp_df['model'] = model_name
                combined_forecasts.append(temp_df)

            # forecast_result가 다른 형태인 경우 (딕셔너리 등)
            elif isinstance(forecast_data, dict):
                # 예측값이 포함된 키를 찾아서 처리
                if 'forecast' in forecast_data:
                    temp_df = pd.DataFrame(forecast_data['forecast'])
                    temp_df['model'] = model_name
                    combined_forecasts.append(temp_df)

        except Exception as e:
            print(f"모델 {model_name} 결과 통합 중 오류: {e}")
            continue

    if combined_forecasts:
        return pd.concat(combined_forecasts, ignore_index=True)
    else:
        return pd.DataFrame()

# 통합된 예측 결과 생성
combined_forecast_df = combine_all_forecasts(all_forecast_results)

# 4. 결과 데이터를 컬럼별로 통합하는 방법
# =================================================================

def merge_forecast_columns(forecast_results_dict, base_data):
    """
    각 모델의 예측 결과를 base_data에 새로운 컬럼으로 추가
    """
    result_df = base_data.copy()

    for model_name, results in forecast_results_dict.items():
        try:
            forecast_data = results['forecast_result']

            # 예측 컬럼명 설정
            forecast_col = f'forecast_{model_name}'
            confidence_lower_col = f'forecast_{model_name}_lower'
            confidence_upper_col = f'forecast_{model_name}_upper'

            # forecast_data에서 예측값 추출하여 병합
            if isinstance(forecast_data, pd.DataFrame):
                # 날짜 기준으로 병합
                if 'date' in forecast_data.columns:
                    merge_df = forecast_data[['date', 'forecast']].copy()
                    merge_df.columns = ['date', forecast_col]
                    result_df = pd.merge(result_df, merge_df, on='date', how='left')

                    # 신뢰구간이 있는 경우 추가
                    if 'forecast_lower' in forecast_data.columns:
                        lower_df = forecast_data[['date', 'forecast_lower']].copy()
                        lower_df.columns = ['date', confidence_lower_col]
                        result_df = pd.merge(result_df, lower_df, on='date', how='left')

                    if 'forecast_upper' in forecast_data.columns:
                        upper_df = forecast_data[['date', 'forecast_upper']].copy()
                        upper_df.columns = ['date', confidence_upper_col]
                        result_df = pd.merge(result_df, upper_df, on='date', how='left')

        except Exception as e:
            print(f"모델 {model_name} 컬럼 병합 중 오류: {e}")
            continue

    return result_df

# base_data는 원본 데이터 (result 변수)
final_combined_data = merge_forecast_columns(all_forecast_results, result)

# 5. 모델 성능 비교를 위한 요약 테이블 생성
# =================================================================

def create_model_summary(forecast_results_dict):
    """
    모든 모델의 성능 지표를 요약한 테이블 생성
    """
    summary_data = []

    for model_name, results in forecast_results_dict.items():
        try:
            model_info = results['model_info']

            summary_row = {'model': model_name}

            # 모델 정보에서 성능 지표 추출
            if isinstance(model_info, dict):
                # 일반적인 성능 지표들
                metrics = ['mse', 'rmse', 'mae', 'mape', 'aic', 'bic', 'r2']
                for metric in metrics:
                    if metric in model_info:
                        summary_row[metric] = model_info[metric]

            summary_data.append(summary_row)

        except Exception as e:
            print(f"모델 {model_name} 요약 생성 중 오류: {e}")
            continue

    return pd.DataFrame(summary_data)

# 모델 성능 요약 테이블
model_performance_summary = create_model_summary(all_forecast_results)

# 6. 결과 확인 및 저장
# =================================================================

print("=== 모든 예측 모델 결과 저장 완료 ===")
print(f"SARIMA 결과: sarima_result_data, sarima_forecast_result")
print(f"LSTM 결과: lstm_result_data, lstm_forecast_result")
print(f"Prophet 결과: prophet_result_data, prophet_forecast_results")
print(f"ES 결과: es_result_data, es_forecast_result")
print(f"통합 결과: all_forecast_results (딕셔너리)")
print(f"통합 예측 DataFrame: combined_forecast_df")
print(f"컬럼별 통합 데이터: final_combined_data")
print(f"모델 성능 요약: model_performance_summary")

# 각 모델 결과에 개별적으로 접근하는 방법
print("\n=== 개별 모델 결과 접근 방법 ===")
print("SARIMA 예측 결과:", type(all_forecast_results['sarima']['forecast_result']))
print("LSTM 예측 결과:", type(all_forecast_results['lstm']['forecast_result']))
print("Prophet 예측 결과:", type(all_forecast_results['prophet']['forecast_result']))
print("ES 예측 결과:", type(all_forecast_results['exponential_smoothing']['forecast_result']))

# 7. 선택적 저장 (필요한 경우)
# =================================================================

# CSV 파일로 저장
# combined_forecast_df.to_csv('combined_forecasts.csv', index=False)
# final_combined_data.to_csv('final_combined_data.csv', index=False)
# model_performance_summary.to_csv('model_performance_summary.csv', index=False)

# Excel 파일로 모든 결과 저장
# with pd.ExcelWriter('forecast_results.xlsx') as writer:
#     combined_forecast_df.to_excel(writer, sheet_name='Combined_Forecasts', index=False)
#     final_combined_data.to_excel(writer, sheet_name='Final_Data', index=False)
#     model_performance_summary.to_excel(writer, sheet_name='Model_Summary', index=False)
#
#     # 각 모델별 상세 결과도 별도 시트로 저장
#     for model_name, results in all_forecast_results.items():
#         try:
#             if isinstance(results['forecast_result'], pd.DataFrame):
#                 results['forecast_result'].to_excel(writer, sheet_name=f'{model_name}_forecast', index=False)
#         except:
#             pass

SARIMA 예측 시작
예측 시작일: 2025-10
외생변수 사용: True
예측 기간: 12개월
예측 종료일: 2026-09
과거 수출 데이터: 153개
미래 수출 예측치: 12개
과거 PSR 데이터: 153개

외생변수(수출 데이터) 준비 중... (YoY 변환)
외생변수(YoY) 매칭된 학습 개월: 141
미래 수출 YoY 예측치 개월: 12

PSR 정상성 검정:
ADF Statistic: -0.3190
p-value: 0.9228
시계열이 비정상적입니다. 차분이 필요할 수 있습니다.

수출 데이터 정상성 검정:
ADF Statistic: -3.9717
p-value: 0.0016
시계열이 정상적입니다.
최적 SARIMA 파라미터 탐색 중...
총 144개 조합 테스트
진행률: 10/144
진행률: 20/144
진행률: 30/144
진행률: 40/144
진행률: 50/144
진행률: 60/144
진행률: 70/144
진행률: 80/144
진행률: 90/144
진행률: 100/144
진행률: 110/144
진행률: 120/144
진행률: 130/144
진행률: 140/144

최적 파라미터:
ARIMA Order: (2, 1, 2)
Seasonal Order: (1, 0, 0, 12)
Best AIC: -28.7040

최종 SARIMA 모델 학습 중...
모델 학습 완료!
AIC: -28.7040

12개월 예측 수행 중...
SARIMA 예측 완료!
예측 결과: 12 레코드
=== 매출 SARIMA 예측 파이프라인 시작 ===
데이터 종료일: 2025-08-31
예측 분기 수: 4

1. 분기별 매출 데이터 추출
데이터를 2025-08까지로 제한했습니다.
유효한 매출 데이터: 151개월
데이터 기간: 2013-01 ~ 2025-08
추출된 분기 데이터: 51분기
분기별 데이터:
  2013Q1: 33.33B (3개월 데이터)
  2013Q2: 34.95B (3개월 데이터)
  2013Q3: 35.30B (3개월 데이터)
  2013Q4: 39.16B 

16:53:18 - cmdstanpy - INFO - Chain [1] start processing


Prophet 모델 훈련 중...


16:53:19 - cmdstanpy - INFO - Chain [1] done processing


분기별 Prophet 예측 완료 (외생변수 미포함):
  2025Q4: -6.81B
  2026Q1: -14.02B
  2026Q2: -5.75B
  2026Q3: -5.58B
외생변수 미포함 Prophet 예측 성공
분기별 Prophet 예측값을 월별로 분배 중...
2025Q4 Prophet 예측값 -6.81B를 [10, 11, 12]월에 동일하게 적용
  -> 2025-10-31: -6.81B
  -> 2025-11-30: -6.81B
  -> 2025-12-31: -6.81B
2026Q1 Prophet 예측값 -14.02B를 [1, 2, 3]월에 동일하게 적용
  -> 2026-01-31: -14.02B
  -> 2026-02-28: -14.02B
  -> 2026-03-31: -14.02B
2026Q2 Prophet 예측값 -5.75B를 [4, 5, 6]월에 동일하게 적용
  -> 2026-04-30: -5.75B
  -> 2026-05-31: -5.75B
  -> 2026-06-30: -5.75B
2026Q3 Prophet 예측값 -5.58B를 [7, 8, 9]월에 동일하게 적용
  -> 2026-07-31: -5.58B
  -> 2026-08-31: -5.58B
  -> 2026-09-30: -5.58B

3. Prophet 예측 (외생변수 포함)
사용 가능한 외생변수: ['expDlr']
외생변수 NaN 제거: 0개 행 제거됨 (전체 152개 중)
추출된 분기별 외생변수 데이터: 51분기
외생변수 데이터 추가 중...
추가할 외생변수: ['expDlr']
추가된 외생변수: ['expDlr']
Prophet 데이터 준비 완료: 51개 분기
Prophet 모델링 시작 (외생변수 포함)


16:53:19 - cmdstanpy - INFO - Chain [1] start processing


외생변수 추가: expDlr
Prophet 모델 훈련 중...


16:53:20 - cmdstanpy - INFO - Chain [1] done processing


외생변수의 미래 값 설정 중...
Prophet 예측 실패: Regressor 'expDlr' missing from dataframe
외생변수 포함 Prophet 예측 실패

=== 매출 Prophet 예측 완료 ===
완료된 예측: 1/2개
추가된 컬럼:
  - revenue_billions_prophet_forecast
=== 매출 Exponential Smoothing 예측 파이프라인 시작 ===
데이터 종료일: 2025-08-31
예측 분기 수: 4

1. 분기별 매출 데이터 추출
데이터를 2025-08까지로 제한했습니다.
유효한 매출 데이터: 151개월
데이터 기간: 2013-01 ~ 2025-08
추출된 분기 데이터: 51분기
분기별 데이터:
  2013Q1: 33.33B (3개월 데이터)
  2013Q2: 34.95B (3개월 데이터)
  2013Q3: 35.30B (3개월 데이터)
  2013Q4: 39.16B (3개월 데이터)
  2014Q1: 39.16B (3개월 데이터)
  2014Q2: 31.92B (2개월 데이터)
  2014Q3: 31.85B (3개월 데이터)
  2014Q4: 42.02B (3개월 데이터)
  2015Q1: 26.10B (3개월 데이터)
  2015Q2: 28.45B (3개월 데이터)
  2015Q3: 27.86B (3개월 데이터)
  2015Q4: 23.52B (3개월 데이터)
  2016Q1: 27.84B (3개월 데이터)
  2016Q2: 30.34B (3개월 데이터)
  2016Q3: 29.04B (3개월 데이터)
  2016Q4: 32.47B (3개월 데이터)
  2017Q1: 26.88B (3개월 데이터)
  2017Q2: 29.10B (3개월 데이터)
  2017Q3: 30.66B (3개월 데이터)
  2017Q4: 32.39B (3개월 데이터)
  2018Q1: 27.79B (3개월 데이터)
  2018Q2: 29.16B (3개월 데이터)
  2018Q3: 23.39B (3개월 데이터)
  2018Q4

Traceback (most recent call last):
  File "C:\Users\MetaM\PycharmProjects\stock_forecast\DATA\us_sarima_forecast.py", line 1378, in prophet_quarterly_forecast
    forecast = model.predict(future_df)
  File "C:\Users\MetaM\PycharmProjects\stock_forecast\.venv\lib\site-packages\prophet\forecaster.py", line 1273, in predict
    df = self.setup_dataframe(df.copy())
  File "C:\Users\MetaM\PycharmProjects\stock_forecast\.venv\lib\site-packages\prophet\forecaster.py", line 300, in setup_dataframe
    raise ValueError(
ValueError: Regressor 'expDlr' missing from dataframe


In [21]:
# 2. 그 다음에 통합 실행
revenue_forecast_result = create_revenue_forecast_result(
    sarima_data=sarima_result_data,
    lstm_data=lstm_result_data,
    prophet_data=prophet_result_data,
    es_data=es_result_data
)

In [22]:
result_with_fixed_ttm = calculate_ttm_with_shift(revenue_forecast_result, shift_months=2)

In [23]:
result_with_fixed_ttm.tail(20)

,ticker,date_month_end,revenue_billions_forecast,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_es_forecast,revenue_ttm_forecast,revenue_ttm_lstm_forecast,revenue_ttm_prophet_forecast,revenue_ttm_es_forecast,revenue_ttm_forecast_shift2m,revenue_ttm_lstm_forecast_shift2m,revenue_ttm_prophet_forecast_shift2m,revenue_ttm_es_forecast_shift2m
154,GE,2025-02-28,10.810000,10.810000,10.810000,10.810000,39.670000,39.670000,39.670000,39.670000,38.700000,38.700000,38.700000,38.700000
155,GE,2025-03-31,9.930000,9.930000,9.930000,9.930000,39.670000,39.670000,39.670000,39.670000,39.670000,39.670000,39.670000,39.670000
156,GE,2025-04-30,9.930000,9.930000,9.930000,9.930000,41.600000,41.600000,41.600000,41.600000,39.670000,39.670000,39.670000,39.670000
157,GE,2025-05-31,9.930000,9.930000,9.930000,9.930000,41.600000,41.600000,41.600000,41.600000,39.670000,39.670000,39.670000,39.670000
158,GE,2025-06-30,11.020000,11.020000,11.020000,11.020000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000
159,GE,2025-07-31,11.020000,11.020000,11.020000,11.020000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000
160,GE,2025-08-31,11.020000,11.020000,11.020000,11.020000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000,41.600000
161,GE,2025-09-30,11.020000,11.020000,11.020000,11.020000,NaN,NaN,NaN,NaN,41.600000,41.600000,41.600000,41.600000
162,AMAT,2025-10-31,14.542772,9.134041,-6.809571,12.361423,14.542772,9.134041,-6.809571,12.361423,NaN,NaN,NaN,NaN
163,AMAT,2025-11-30,14.542772,9.134041,-6.809571,12.361423,14.542772,9.134041,-6.809571,12.361423,NaN,NaN,NaN,NaN


In [14]:
revenue_forecast_result

,ticker,date_month_end,revenue_billions_forecast,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_es_forecast
0,GE,2013-01-31,36.760000,36.76,36.760000,36.760000
1,GE,2013-02-28,36.760000,36.76,36.760000,36.760000
2,GE,2013-03-31,33.330000,33.33,33.330000,33.330000
3,GE,2013-04-30,33.330000,33.33,33.330000,33.330000
4,GE,2013-05-31,33.330000,33.33,33.330000,33.330000
...,...,...,...,...,...,...
160,AMAT,2026-05-31,8.022891,8.75,-5.750676,6.240746
161,AMAT,2026-06-30,8.022891,8.75,-5.750676,6.240746
162,AMAT,2026-07-31,6.946333,8.75,-5.581357,6.073058
163,AMAT,2026-08-31,6.946333,8.75,-5.581357,6.073058


In [12]:
rev_pach_data = stock_prep.fetch_ticker_and_item(db_info, 'AMAT', 'US_fundq')
me_pach_data = stock_prep.fetch_ticker_and_me(db_info, 'AMAT', 'US_fundm')

✅ ticker 'AMAT'에서 554건의 saleq 데이터를 가져왔습니다.
📊 데이터 기간: 2000-01-31 00:00:00 ~ 2025-07-31 00:00:00
✅ ticker 'AMAT'에서 316건의 saleq 데이터를 가져왔습니다.
📊 데이터 기간: 2000-01-31 00:00:00 ~ 2025-04-30 00:00:00


In [13]:
test2

,permno,edate,date,ticker,me
0,14702,NaT,2000-01-31,AMAT,53045.340750
1,14702,NaT,2000-02-29,AMAT,70702.965563
2,14702,NaT,2000-03-31,AMAT,72852.799500
3,14702,NaT,2000-04-28,AMAT,82219.193437
4,14702,NaT,2000-05-31,AMAT,67430.842500
...,...,...,...,...,...
311,14702,2024-12-31,2024-12-31,AMAT,133031.343994
312,14702,2025-01-31,2025-01-31,AMAT,147526.304993
313,14702,2025-02-28,2025-02-28,AMAT,129301.265991
314,14702,2025-03-31,2025-03-31,AMAT,118708.156006


In [28]:
print("Loading base data...")
fundq_df = fetch_table_data(db_info, 'US_fundq')

Loading base data...


KeyboardInterrupt: 

In [7]:
test1

,permno,edate,date,ticker,saleq
0,14702,2000-01-31,2000-01-31,AMAT,1722.190
1,14702,2000-02-29,2000-02-29,AMAT,1722.190
2,14702,2000-03-31,2000-03-31,AMAT,1722.190
3,14702,2000-04-30,2000-04-30,AMAT,2190.031
4,14702,2000-05-31,2000-05-31,AMAT,2190.031
...,...,...,...,...,...
549,14702,2025-03-31,2025-03-31,AMAT,7166.000
550,14702,2025-04-30,2025-04-30,AMAT,7100.000
551,14702,2025-05-31,2025-05-31,AMAT,7100.000
552,14702,2025-06-30,2025-06-30,AMAT,7100.000


In [22]:
print("Loading base data...")
fundm_df = fetch_table_data(db_info, 'US_fundm')

Loading base data...
✅ 'US_fundm' 테이블에서 257676건의 데이터를 가져왔습니다.


In [25]:
fundm_df[fundm_df['ticker'] == 'AMAT'][['date','me']]

,date,me
47949,2000-01-31,53045.340750
47950,2000-02-29,70702.965563
47951,2000-03-31,72852.799500
47952,2000-04-28,82219.193437
47953,2000-05-31,67430.842500
...,...,...
48260,2024-12-31,133031.343994
48261,2025-01-31,147526.304993
48262,2025-02-28,129301.265991
48263,2025-03-31,118708.156006


In [16]:
# 현재 stock_prep 모듈에 어떤 함수들이 있는지 확인
print("stock_prep 모듈의 속성들:")
attributes = [attr for attr in dir(stock_prep) if not attr.startswith('_')]
for attr in attributes:
    print(f"- {attr}")

# 함수인지 확인
print("\n함수들:")
import types
functions = [attr for attr in dir(stock_prep) if isinstance(getattr(stock_prep, attr), types.FunctionType)]
for func in functions:
    print(f"- {func}()")


stock_prep 모듈의 속성들:
- STOCK_FUNCTION_AVAILABLE
- add_revenue_ttm
- calculate_psr_with_shift
- calendar
- collect_export_data
- collect_market_cap_data
- collect_revenue_data
- convert_to_month_end
- create_engine
- datetime
- fetch_market_data_yearly
- fetch_revenue_data
- get_db_host
- get_hs_data
- get_latest_input_date_data
- merge_revenue_market_data
- merge_with_export_data
- pd
- process_daily_to_monthly_market_data
- requests
- run_data_preprocessing
- run_simple_preprocessing
- test_api_connection
- time
- tqdm
- warnings

함수들:
- add_revenue_ttm()
- calculate_psr_with_shift()
- collect_export_data()
- collect_market_cap_data()
- collect_revenue_data()
- convert_to_month_end()
- create_engine()
- fetch_market_data_yearly()
- fetch_revenue_data()
- get_db_host()
- get_hs_data()
- get_latest_input_date_data()
- merge_revenue_market_data()
- merge_with_export_data()
- process_daily_to_monthly_market_data()
- run_data_preprocessing()
- run_simple_preprocessing()
- test_api_connectio